In [1]:
function range(n: number): number[] {
    return Array.from({ length: n + 1 }, (_, i) => i);
}

# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals,
- eliminate conditionals,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,
- rename bound variables to avoid clashes, 
- transform the formula into *prenex normal form*,
  i.e. we move the quantifieres outside,
  
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *clauses* in set notation.

When converting formulas into conjunctive normal form, we <u>assume</u> that the formulas are 
*pure*, where we define a formula $f$ as *pure* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall X: q(X)\bigr)$$
is **not** *pure*, because there are two different universal quantifiers that both bind the same variable $X$.  We can rewrite this formulas as a *pure* formula by *renaming* all occurrences of $X$ that are bound by the second quantifier as follows:
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall Y: q(Y)\bigr)$$

## Auxilliary Functions

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the parser that is found in the module `FOL-Parser`.  Our parser distinguishes variables and function symbol as follows:
- A word starting with an *upper* case letter is interpreted as a *variable*.
- A word starting with a *lower* case letter is assumed to be a *function* or *predicate symbol*.

In [2]:
import { parseFormula as parse, Formula, Term, Variable } from "./FOL-Parser";
import { Tuple, RecursiveSet as Set, Value } from "recursive-set";

In [3]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

In [4]:
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

In [5]:
const s = '∀G:∀C:(grandparent(G, C) ↔ ∃P: (parent(G, P) ∧ parent(P, C)))';
const f1 = parse(s);
console.log(f1.toString());

(∀, G, (∀, C, (↔, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C))))))


A `Substitution` maps variables to terms. 

In [6]:
type Substitution = Map<Variable, Term>;

The function $\texttt{apply}(t, σ)$ takes an object $t$ and a *variable substitution* $\sigma$ which is represented as a dictionary of the form $\{X_1: s_1, \cdots, X_n:s_n\}$ and replaces every occurrence of the variable $X_i$ in the object $t$ with the corresponding term $s_i$.  The object $t$ is either 
 - a term, or
 - a formula from first order logic (henceforth abbreviated as *FOL*), 
 - a clause (represented as a `frozenset` of literals), or 
 - a set of clauses.

In [7]:
function applyTerm(t: Term, sigma: Substitution): Term {
    // Base case: if it's a variable (string), substitute or return as is
    if (typeof t === 'string') {
        return sigma.has(t) ? (sigma.get(t) as Term) : t;
    }
    // Spread the Tuple into a standard array and destructure the first element as 
    // 'f' and the rest as 'restArgs'.
    const [f, ...restArgs] = [...t];
    const args = restArgs.map(arg => applyTerm(arg as Term, sigma));
    return tpl(f, ...args) as Term;
}

In [8]:
function applyFormula(f: Formula, sigma: Substitution): Formula {
    const [op, ...args] = [...f];
    switch (op) {
        case '⊤':
        case '⊥':
            return f;
        case '¬':
            return tpl('¬', applyFormula(args[0] as Formula, sigma)) as Formula;
        case '∧':
        case '∨':
        case '→':
        case '↔':
            return tpl(
                op,
                applyFormula(args[0] as Formula, sigma),
                applyFormula(args[1] as Formula, sigma)
            ) as Formula;
        case '∀':
        case '∃': 
            const x = args[0] as string;
            const newX = sigma.has(x) ? (sigma.get(x) as string) : x;
            return tpl(op, newX, applyFormula(args[1] as Formula, sigma)) as Formula;
        default: 
            const subArgs = args.map(arg => applyTerm(arg as Term, sigma));
            return tpl(op, ...subArgs) as Formula;
    }
}

In [9]:
console.log(f1.toString());

(∀, G, (∀, C, (↔, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C))))))


In [10]:
const sigma1: Substitution = new Map([
    ['G', 'X'],
    ['P', 'Y'],
    ['C', 'Z']
]);
console.log(applyFormula(f1, sigma1).toString());

(∀, X, (∀, Z, (↔, (grandparent, X, Z), (∃, Y, (∧, (parent, X, Y), (parent, Y, Z))))))


The function $\texttt{boundVariables}(f)$ computes the set of variables that are *bound* in the formula $f$. 

In [11]:
function boundVariables(f: Formula): Set<string> {
    const op = f.get(0);
    if (op == '∀' || op == '∃') {
        const x = f.get(1) as string;
        const g = f.get(2) as Formula;
        return boundVariables(g).union(set<string>(x));
    }
    if (op == '⊤' || op == '⊥') return set<string>();
    if (op == '¬') return boundVariables(f.get(1) as Formula);
    if (op == '∧' || op == '∨' || op == '→' || op == '↔') {
        return boundVariables(f.get(1) as Formula).union(boundVariables(f.get(2) as Formula));
    }
    return set<string>();
}

In [12]:
console.log(f1.toString());

(∀, G, (∀, C, (↔, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C))))))


In [13]:
console.log([...boundVariables(f1)]);

[ 'P', 'C', 'G' ]


The function `allVariables` computes the set of all variables that occur in terms inside `f`. The object 
`f` is either a formula or a term.

In [14]:
function allVariablesTerm(t: Term): Set<string> {
    // If it's a string, we assume it's a variable if it starts with an uppercase letter.
    // Our logic parser guarantees valid string formats.
    if (typeof t == 'string') return set<string>(t);
    
    let vars = set<string>();
    for (let i = 1; i < t.length; i++) {
        vars = vars.union(allVariablesTerm(t.get(i) as Term));
    }
    return vars;
}

function allVariables(f: Formula): Set<string> {
    const op = f.get(0);
    if (op == '∀' || op == '∃') {
        const x = f.get(1) as string;
        return allVariables(f.get(2) as Formula).union(set<string>(x));
    }
    if (op == '⊤' || op == '⊥') return set<string>();
    if (op == '¬') return allVariables(f.get(1) as Formula);
    if (op == '∧' || op == '∨' || op == '→' || op == '↔') {
        return allVariables(f.get(1) as Formula).union(allVariables(f.get(2) as Formula));
    }
    
    let vars = set<string>();
    for (let i = 1; i < f.length; i++) {
        vars = vars.union(allVariablesTerm(f.get(i) as Term));
    }
    return vars;
}

In [15]:
console.log(f1.toString());

(∀, G, (∀, C, (↔, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C))))))


In [16]:
console.log([...allVariables(f1)]);

[ 'G', 'C', 'P' ]


In [17]:
const g1 = tpl('↔', 
    tpl('grandparent', 'G', 'C'),
    tpl('∃', 'P', tpl('∧', tpl('parent', 'G', 'P'), tpl('parent', 'P', 'C')))
) as Formula;

In [18]:
console.log([...allVariables(g1)]);

[ 'G', 'C', 'P' ]


Below we construct a list of all upper case characters to generate new variables.

In [19]:
const ascii_uppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ".split("");
console.log(ascii_uppercase.join(""));

ABCDEFGHIJKLMNOPQRSTUVWXYZ


In [20]:
const ascii_set = set<string>(...ascii_uppercase);
console.log(ascii_set.size);

26


The function $\texttt{renameBoundVariables}(f)$ takes a first order formula $f$ and replaces all bound variables by **new** variables.  This only works if the set of characters `set(string.ascii_uppercase)` has enough characters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `X1`, `X2`, `X3`, $\cdots$, but that becomes unreadable very fast.

In [21]:
function renameBoundVariables(f: Formula): Formula {
    const boundVs = [...boundVariables(f)];
    const allVs = allVariables(f);
    
    const newVars = ascii_uppercase.filter(x => !allVs.has(x)).sort();
    
    const sigma: Substitution = new Map();
    for (let i = 0; i < boundVs.length; i++) {
        sigma.set(boundVs[i], newVars[i]);
    }
    
    return applyFormula(f, sigma);
}

In [22]:
console.log(['A', 'B', 'C'].map((x, i) => [i, x]));

[ [ 0, 'A' ], [ 1, 'B' ], [ 2, 'C' ] ]


In [23]:
console.log(f1.toString());

(∀, G, (∀, C, (↔, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C))))))


In [24]:
console.log(renameBoundVariables(f1).toString());

(∀, D, (∀, B, (↔, (grandparent, D, B), (∃, A, (∧, (parent, D, A), (parent, A, B))))))


## Elimination Biconditionals

The function $\texttt{eliminateBiconditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$ (f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f) $$
In order to ensure that the resulting formula is <em style="color:blue">pure</em>, we have to rename the bound variables in the formula $g \rightarrow f$.

In [25]:
function eliminateBiconditional(f: Formula): Formula {
    const op = f.get(0);
    if (op == '↔') {
        const g = f.get(1) as Formula;
        const h = f.get(2) as Formula;
        const ge = eliminateBiconditional(g);
        const he = eliminateBiconditional(h);
        const left = tpl('→', ge, he) as Formula;
        const right = renameBoundVariables(tpl('→', he, ge) as Formula);
        return tpl('∧', left, right) as Formula;
    }
    if (op == '⊤' || op == '⊥') return f;
    if (op == '¬') return tpl('¬', eliminateBiconditional(f.get(1) as Formula)) as Formula;
    if (op == '∧' || op == '∨' || op == '→') {
        return tpl(op, eliminateBiconditional(f.get(1) as Formula), eliminateBiconditional(f.get(2) as Formula)) as Formula;
    }
    if (op == '∀' || op == '∃') {
        return tpl(op, f.get(1) as string, eliminateBiconditional(f.get(2) as Formula)) as Formula;
    }
    return f;
}

In [26]:
console.log(f1.toString());

(∀, G, (∀, C, (↔, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C))))))


In [27]:
const f2 = eliminateBiconditional(f1);
console.log(f2.toString());

(∀, G, (∀, C, (∧, (→, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C)))), (→, (∃, A, (∧, (parent, G, A), (parent, A, C))), (grandparent, G, C)))))


## Eliminating Conditionals

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$ (f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g) $$
The implementation of this function is similar to the implementation of the function `eliminateConditional` that we had used in propositional logic.

In [28]:
function eliminateConditional(f: Formula): Formula {
    const op = f.get(0);
    if (op == '→') {
        const g = f.get(1) as Formula;
        const h = f.get(2) as Formula;
        return tpl('∨', tpl('¬', eliminateConditional(g)), eliminateConditional(h)) as Formula;
    }
    if (op == '⊤' || op == '⊥') return f;
    if (op == '¬') return tpl('¬', eliminateConditional(f.get(1) as Formula)) as Formula;
    if (op == '∧' || op == '∨') {
        return tpl(op, eliminateConditional(f.get(1) as Formula), eliminateConditional(f.get(2) as Formula)) as Formula;
    }
    if (op == '∀' || op == '∃') {
        return tpl(op, f.get(1) as string, eliminateConditional(f.get(2) as Formula)) as Formula;
    }
    return f;
}

In [29]:
console.log(f2.toString());

(∀, G, (∀, C, (∧, (→, (grandparent, G, C), (∃, P, (∧, (parent, G, P), (parent, P, C)))), (→, (∃, A, (∧, (parent, G, A), (parent, A, C))), (grandparent, G, C)))))


In [30]:
const f3 = eliminateConditional(f2);
console.log(f3.toString());

(∀, G, (∀, C, (∧, (∨, (¬, (grandparent, G, C)), (∃, P, (∧, (parent, G, P), (parent, P, C)))), (∨, (¬, (∃, A, (∧, (parent, G, A), (parent, A, C)))), (grandparent, G, C)))))


## Negation Normal Form

The function $\texttt{nnf}(f)$ computes the <em style="color:blue;">negation normal form</em> of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$.  The expression $\texttt{nnf}(f)$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
    <li> $\texttt{nnf}(\forall x: F) = \forall x: \texttt{nnf}(\texttt{F})$.</li>
    <li> $\texttt{nnf}(\exists x: F) = \exists x: \texttt{nnf}(\texttt{F})$.</li>
</ol>

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

In [31]:
function nnf(f: Formula): Formula {
    const op = f.get(0);
    if (op == '⊤' || op == '⊥') return f;
    if (op == '¬') return neg(f.get(1) as Formula);
    if (op == '∧' || op == '∨') {
        return tpl(op, nnf(f.get(1) as Formula), nnf(f.get(2) as Formula)) as Formula;
    }
    if (op == '∀' || op == '∃') {
        return tpl(op, f.get(1) as string, nnf(f.get(2) as Formula)) as Formula;
    }
    return f;
}

function neg(f: Formula): Formula {
    const op = f.get(0);
    if (op == '⊤') return tpl('⊥') as Formula;
    if (op == '⊥') return tpl('⊤') as Formula;
    if (op == '¬') return nnf(f.get(1) as Formula);
    if (op == '∧') return tpl('∨', neg(f.get(1) as Formula), neg(f.get(2) as Formula)) as Formula;
    if (op == '∨') return tpl('∧', neg(f.get(1) as Formula), neg(f.get(2) as Formula)) as Formula;
    if (op == '∀') return tpl('∃', f.get(1) as string, neg(f.get(2) as Formula)) as Formula;
    if (op == '∃') return tpl('∀', f.get(1) as string, neg(f.get(2) as Formula)) as Formula;
    return tpl('¬', f) as Formula;
}

In [32]:
console.log(f3.toString());

(∀, G, (∀, C, (∧, (∨, (¬, (grandparent, G, C)), (∃, P, (∧, (parent, G, P), (parent, P, C)))), (∨, (¬, (∃, A, (∧, (parent, G, A), (parent, A, C)))), (grandparent, G, C)))))


In [33]:
const f4 = nnf(f3);
console.log(f4.toString());

(∀, G, (∀, C, (∧, (∨, (¬, (grandparent, G, C)), (∃, P, (∧, (parent, G, P), (parent, P, C)))), (∨, (∀, A, (∨, (¬, (parent, G, A)), (¬, (parent, A, C)))), (grandparent, G, C)))))


## Prenex Normal Form

In the following we assume that all quantifiers that occur in a formula bind **different** variables, i.e. we assume that the formulas are *pure*.  If this assumption is not satisfied, then the functions given below will produce <u>garbage</u>.

A *quantifier tuple* is a tuple of the following form:
$$ (Q_1, X_1, \cdots, Q_n, X_n) $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $X_i$ are variables.  The function $\texttt{mergeQuantifiers}(T_1, T_2)$ takes two quantifier tuples $T_1$ and $T_2$ as arguments and merges them into a new quantifier tuple such that the relative order of the quantifiers remains the same, i.e. if both $Q_1, X_1$ and $Q_2, X_2$ occur in $T_1$ and $Q_1, X_1$ occurs before $Q_2, X_2$, then $Q_1, X_1$ will occur before $Q_2, X_2$ in the result.

In [34]:
type QuantifierList = string[];

function mergeQuantifiers(Q1: QuantifierList, Q2: QuantifierList): QuantifierList {
    if (Q1.length == 0) return Q2;
    if (Q2.length == 0) return Q1;
    if (Q1[0] == '∃') return [Q1[0], Q1[1], ...mergeQuantifiers(Q1.slice(2), Q2)];
    if (Q2[0] == '∃') return [Q2[0], Q2[1], ...mergeQuantifiers(Q1, Q2.slice(2))];
    return [Q1[0], Q1[1], ...mergeQuantifiers(Q1.slice(2), Q2)];
}

In [35]:
console.log(mergeQuantifiers(['∀', 'X', '∃', 'Y'], ['∃', 'U', '∀', 'V']));

[
  '∃', 'U', '∀',
  'X', '∃', 'Y',
  '∀', 'V'
]


Given a formula $f$, the function $\texttt{extractQuantifiers}(f)$ returns a pairs $(T, m)$, where $T$ is a quantifier tuple and $m$ is the <em style="color:blue;">matrix</em> of the formula $f$, where the matrix of a formula is defined as the part that remains when all quantifiers have been extracted.

In [36]:
function extractQuantifiers(f: Formula): [QuantifierList, Formula] {
    const op = f.get(0);
    if (op == '⊤' || op == '⊥' || op == '¬') return [[], f];
    if (op == '∧' || op == '∨') {
        const [qg, gm] = extractQuantifiers(f.get(1) as Formula);
        const [qh, hm] = extractQuantifiers(f.get(2) as Formula);
        return [mergeQuantifiers(qg, qh), tpl(op, gm, hm) as Formula];
    }
    if (op == '∀' || op == '∃') {
        const x = f.get(1) as string;
        const [qg, gm] = extractQuantifiers(f.get(2) as Formula);
        return [[op as string, x, ...qg], gm];
    }
    return [[], f];
}

In [37]:
console.log(f4.toString());

(∀, G, (∀, C, (∧, (∨, (¬, (grandparent, G, C)), (∃, P, (∧, (parent, G, P), (parent, P, C)))), (∨, (∀, A, (∨, (¬, (parent, G, A)), (¬, (parent, A, C)))), (grandparent, G, C)))))


In [38]:
const [Qs, f5] = extractQuantifiers(f4);
console.log(Qs, f5.toString());

[
  '∀', 'G', '∀',
  'C', '∃', 'P',
  '∀', 'A'
] (∧, (∨, (¬, (grandparent, G, C)), (∧, (parent, G, P), (parent, P, C))), (∨, (∨, (¬, (parent, G, A)), (¬, (parent, A, C))), (grandparent, G, C)))


Given a qantifier tuple $\texttt{Qs}$ and a matrix $m$, the call $\texttt{attachQuantifiers}(Qs, m)$ combines the quantifiers $\texttt{Qs}$ and the matrix $m$ into a quantified formula.

In [39]:
function attachQuantifiers(Qs: QuantifierList, m: Formula): Formula {
    if (Qs.length == 0) return m;
    const Q = Qs[0];
    const x = Qs[1];
    return tpl(Q, x, attachQuantifiers(Qs.slice(2), m)) as Formula;
}

In [40]:
console.log(Qs);

[
  '∀', 'G', '∀',
  'C', '∃', 'P',
  '∀', 'A'
]


In [41]:
console.log(f5.toString());

(∧, (∨, (¬, (grandparent, G, C)), (∧, (parent, G, P), (parent, P, C))), (∨, (∨, (¬, (parent, G, A)), (¬, (parent, A, C))), (grandparent, G, C)))


In [42]:
const f6 = attachQuantifiers(Qs, f5);
console.log(f6.toString());

(∀, G, (∀, C, (∃, P, (∀, A, (∧, (∨, (¬, (grandparent, G, C)), (∧, (parent, G, P), (parent, P, C))), (∨, (∨, (¬, (parent, G, A)), (¬, (parent, A, C))), (grandparent, G, C)))))))


## Skolemization (Eliminating Existential Quantifiers)

The variable $\texttt{skolemCounter}$ is a global variable that is needed to create unique Skolem constants.  

In [43]:
let skolemCounter = 0;

In [44]:
function skolemConstant(): string {
    skolemCounter += 1;
    // Generates sk1, sk2, etc. which are valid lowercase function symbols.
    return 'sk' + skolemCounter.toString();
}

The function $\texttt{skolemize}(f, \texttt{Vs})$ takes a formula $f$ and a tuple of variables $\texttt{Vs}$ and 
<em style="color:blue">skolemizes</em> the formula $f$, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The tuple $\texttt{Vs}$ is a tuple of variables that are 
assumed to be universally quantified.  The formula $f$ is assumed to be in <em style="color:blue">prenex normal form</em>.

For skolemization to work correctly, we have to assume that 
<font size="4" style="color:darkgreen; size:125%">$f$ does not contain free variables</font>!

In [45]:
function skolemize(f: Formula, Vs: string[]): Formula {
    const op = f.get(0);
    if (op == '∃') {
        const x = f.get(1) as string;
        const g = f.get(2) as Formula;
        const t = tpl(skolemConstant(), ...Vs) as Term;
        const sigma: Substitution = new Map();
        sigma.set(x, t);
        return skolemize(applyFormula(g, sigma), Vs);
    }
    if (op == '∀') {
        const x = f.get(1) as string;
        const g = f.get(2) as Formula;
        return skolemize(g, [...Vs, x]);
    }
    return f;
}

In [46]:
console.log(f6.toString());

(∀, G, (∀, C, (∃, P, (∀, A, (∧, (∨, (¬, (grandparent, G, C)), (∧, (parent, G, P), (parent, P, C))), (∨, (∨, (¬, (parent, G, A)), (¬, (parent, A, C))), (grandparent, G, C)))))))


In [47]:
const f7 = skolemize(f6, []);
console.log(f7.toString());

(∧, (∨, (¬, (grandparent, G, C)), (∧, (parent, G, (sk1, G, C)), (parent, (sk1, G, C), C))), (∨, (∨, (¬, (parent, G, A)), (¬, (parent, A, C))), (grandparent, G, C)))


## Conversion to Clauses

The function $\texttt{cnf}(f)$ takes a <em style="color:blue">skolemized</em> formula $f$ from first order logic that is in <em style="color:blue">negation normal form</em> and returns the <em style="color:blue">conjunctive normal form</em> of $f$ in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic.

In [48]:
type Literal = Formula;
type Clause = Set<Literal>;
type CNFSet = Set<Clause>;

function cnf(f: Formula): CNFSet {
    const op = f.get(0);
    if (op == '⊤') return set<Clause>();
    if (op == '⊥') return set<Clause>(set<Literal>());
    if (op == '¬') return set<Clause>(set<Literal>(f));
    if (op == '∧') {
        const g = f.get(1) as Formula;
        const h = f.get(2) as Formula;
        return cnf(g).union(cnf(h));
    }
    if (op == '∨') {
        const g = f.get(1) as Formula;
        const h = f.get(2) as Formula;
        const cnfg = cnf(g);
        const cnfh = cnf(h);
        const result = set<Clause>();
        for (const k1 of cnfg) {
            for (const k2 of cnfh) {
                result.add(k1.union(k2));
            }
        }
        return result;
    }
    return set<Clause>(set<Literal>(f));
}

In [49]:
console.log(f7.toString());

(∧, (∨, (¬, (grandparent, G, C)), (∧, (parent, G, (sk1, G, C)), (parent, (sk1, G, C), C))), (∨, (∨, (¬, (parent, G, A)), (¬, (parent, A, C))), (grandparent, G, C)))


In [50]:
const f8 = cnf(f7);
console.log(f8.size);

3


## Putting Everything Together

The function $f$ takes a <em style="color:blue">pure</em> formula $f$ from first order logic and transforms $f$ into a set of first order clauses.  Furthermore, $f$ **must not** contain free variables.

In [51]:
function normalize(f: Formula): CNFSet {
    const f1 = eliminateBiconditional(f);
    const f2 = eliminateConditional(f1);
    const f3 = nnf(f2);
    const [Qs, f4] = extractQuantifiers(f3);
    const f5 = attachQuantifiers(Qs, f4);
    const f6 = skolemize(f5, []);
    return cnf(f6);
}

In [52]:
console.log(normalize(f1).size);

3


In [53]:
function test(s: string): void {
    const f = parse(s);
    console.log(`The knf of ${s} is:`);
    console.log(normalize(f));
}

In [54]:
function prettify(M: CNFSet): string {
    if (M.size == 0) return '{}';
    let result = "{\n";
    const clauses = [...M];
    for (let i = 0; i < clauses.length; i++) {
        const A = clauses[i];
        if (A.size == 0) {
            result += "    {},\n";
        } else {
            result += "    {" + [...A].map(lit => lit.toString()).join(", ") + "}";
            if (i < clauses.length - 1) result += ",\n";
            else result += "\n";
        }
    }
    result += "}";
    return result;
}

In [55]:
test(s);

The knf of ∀G:∀C:(grandparent(G, C) ↔ ∃P: (parent(G, P) ∧ parent(P, C))) is:
{{(grandparent, G, C), (¬, (parent, A, C)), (¬, (parent, G, A))}, {(parent, (sk3, G, C), C), (¬, (grandparent, G, C))}, {(parent, G, (sk3, G, C)), (¬, (grandparent, G, C))}}


In [56]:
test('¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V))');

The knf of ¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V)) is:
{{(p, X, (sk4))}, {(¬, (p, (sk5), V))}}
